# 04 - SHAP decision plots for representative ecoprovinces (Fig. 5d)

**Run order:** `01_fit_full_model.py` -> `03_shap_maps.ipynb` -> **`04_decision_plots.ipynb`**.

**Purpose.** For every USDA Forest Service ecoprovince, pick one representative grid cell: among the
cells whose NLCD majority class is the ecoprovince's most common class, the one closest (Euclidean
distance) to the ecoprovince mean of GPP and all model predictors. For a manually chosen set of 15
ecoprovinces spanning the humid East and the arid West, draw a decision plot: starting from the output
of the baseline model (0), cumulatively add the SHAP values of PC1, PC2, PC3, FRic_gamma,
Fbeta_alpha_to_gamma, Fbeta_gamma_to_tau and FDiv_gamma. Line colour encodes the residual of the
baseline model (GPP - mean GPP - SHAP(LAI, T, P, AI)); point colour encodes the value of the predictor
being added (rank among the 15 cells); marker shape encodes the NLCD class of the cell.

**Inputs**
- `./results/shap_values_original_scale.csv` - written by 03 (one row per valid grid cell)

**Outputs**
- `./results/fig5d_selected_ecoprovinces.csv` - the 15 representative cells with their cumulative SHAP values
- `./results/fig5d_decision_plots.png` - Fig. 5d

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial.distance import cdist

RESULTS_DIR = './results'
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
null_list = ['LAI', 'Temperature', 'Precipitation', 'aridity_index']
null_SHAP_list = [x + '_SHAP' for x in null_list]
trait_list = ['all_PC1', 'all_PC2', 'all_PC3']
trait_SHAP_list = [x + '_SHAP' for x in trait_list]
FD_list = ['FRic_gamma', 'Fbeta_alpha_to_gamma', 'Fbeta_gamma_to_tau', 'FDiv_gamma']
FD_SHAP_list = [x + '_SHAP' for x in FD_list]

In [ ]:
# ecoprovince codes such as '211' must stay strings
shap_values_df = pd.read_csv(f'{RESULTS_DIR}/shap_values_original_scale.csv', dtype={'ecoprovince': str})

In [ ]:
# drop cells with a missing response or predictor
full_model_var_list = ['all_PC1', 'all_PC2', 'all_PC3', 'FRic_gamma', 'Fbeta_alpha_to_gamma', 'Fbeta_gamma_to_tau',
                       'FDiv_gamma', ]
shap_values_df = shap_values_df.dropna(
    subset=['GPP', 'LAI', 'Temperature', 'Precipitation', 'aridity_index'] + full_model_var_list)

# Representative grid cell of each ecoprovince

In [ ]:
# drop cells without an ecoprovince label
shap_values_df = shap_values_df.dropna(subset=['ecoprovince'])

In [ ]:
selected_columns = [
    "GPP",
    "LAI",
    "Temperature",
    "Precipitation",
    "aridity_index",
] + full_model_var_list
group_means = shap_values_df.groupby("ecoprovince")[selected_columns].mean()

closest_rows = []

# for each ecoprovince, pick the cell closest to the ecoprovince mean
for group, group_df in shap_values_df.groupby("ecoprovince"):
    mean_values = group_means.loc[group].values.reshape(1, -1)
    # restrict to cells in the ecoprovince's most common NLCD class
    group_df_majority = group_df.loc[
        group_df["nlcd_majority"] == group_df["nlcd_majority"].mode().values[0]
    ]
    # Euclidean distance of every candidate cell to the mean
    distances = cdist(
        group_df_majority[selected_columns], mean_values, metric="euclidean"
    )
    closest_idx = distances.argmin()
    closest_rows.append(group_df_majority.iloc[closest_idx])

shap_values_df_sample = pd.DataFrame(closest_rows)
# the two wetland classes share one marker in the figure
shap_values_df_sample.replace(
    {"Emergent Herbaceous Wetlands": "Woody Wetlands"}, inplace=True
)
shap_values_df_sample.head()

In [ ]:
shap_values_df_sample_v3 = shap_values_df_sample.copy()

In [ ]:
# cumulative SHAP of the trait / FD predictors, starting from the baseline-model output (0)
shap_values_df_sample_v3.insert(1, 'SHAP_base', 0)
shap_values_df_sample_v3.rename(columns={'all_PC1_SHAP': 'SHAP_PC1'}, inplace=True)
shap_values_df_sample_v3['SHAP_PC1+PC2'] = shap_values_df_sample_v3['SHAP_PC1'] + shap_values_df_sample_v3[
    'all_PC2_SHAP']
shap_values_df_sample_v3['SHAP_PC1+PC2+PC3'] = shap_values_df_sample_v3['SHAP_PC1+PC2'] + shap_values_df_sample_v3[
    'all_PC3_SHAP']
shap_values_df_sample_v3['SHAP_PC1+PC2+PC3+FRic-gamma'] = shap_values_df_sample_v3['SHAP_PC1+PC2+PC3'] + shap_values_df_sample_v3[
    'FRic_gamma_SHAP']
shap_values_df_sample_v3['SHAP_PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma'] = shap_values_df_sample_v3[
    'SHAP_PC1+PC2+PC3+FRic-gamma'] + shap_values_df_sample_v3['Fbeta_alpha_to_gamma_SHAP']
shap_values_df_sample_v3['SHAP_PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma+Fbeta-gamma-to-tau'] = \
    shap_values_df_sample_v3['SHAP_PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma'] + \
    shap_values_df_sample_v3['Fbeta_gamma_to_tau_SHAP']
shap_values_df_sample_v3['SHAP_PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma+Fbeta-gamma-to-tau+FDiv-gamma'] = \
    shap_values_df_sample_v3['SHAP_PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma+Fbeta-gamma-to-tau'] + \
    shap_values_df_sample_v3['FDiv_gamma_SHAP']

# residual of the baseline model: GPP - mean GPP - SHAP(LAI, T, P, AI)
shap_values_df_sample_v3['null_SHAP_sum'] = shap_values_df_sample_v3[null_SHAP_list].sum(axis=1)
shap_values_df_sample_v3['base_value'] = shap_values_df_sample_v3['GPP'].mean()
shap_values_df_sample_v3['Residual_null_model'] = shap_values_df_sample_v3['GPP'] - shap_values_df_sample_v3['base_value'] - shap_values_df_sample_v3[
    'null_SHAP_sum']

In [ ]:
# long format: one row per (ecoprovince, cumulative step)
shap_values_df_sample_v3_long = pd.wide_to_long(shap_values_df_sample_v3, stubnames='SHAP', i='ecoprovince',
                                                 j='feature', sep='_', suffix='.*').reset_index()
shap_values_df_sample_v3_long.head()

# Manual selection of ecoprovinces

In [ ]:
# the 15 ecoprovinces shown in Fig. 5d ('212' was listed twice in the original list; isin() ignores the duplicate)
selected_ecoprovinces = ['M221', '211', '212', '222', 'M223', '212', '223', '234', 'M331', '251', '262', '411', '322',
                         '321', 'M333', 'M334']
shap_values_df_sample_manual = shap_values_df_sample_v3[shap_values_df_sample_v3['ecoprovince'].isin(selected_ecoprovinces)]

In [ ]:
shap_values_df_sample_manual.to_csv(f'{RESULTS_DIR}/fig5d_selected_ecoprovinces.csv', index=False)

In [ ]:
shap_values_df_sample_manual_long = pd.wide_to_long(shap_values_df_sample_manual, stubnames='SHAP', i='ecoprovince',
                                                    j='feature', sep='_', suffix='.*').reset_index()

In [ ]:
# point colour at each step = rank of the predictor being added among the selected ecoprovinces
# (RdPu for the functional composition PCs, YlGnBu for the functional diversity metrics)
n_sel = shap_values_df_sample_manual.shape[0]
PC1_color_dict = dict(zip(shap_values_df_sample_manual_long.loc[
                              shap_values_df_sample_manual_long['feature'] == 'PC1'].sort_values('all_PC1')['ecoprovince'],
                          plt.get_cmap('RdPu')(np.linspace(0.2, 0.9, n_sel)).tolist()))
PC2_color_dict = dict(zip(shap_values_df_sample_manual_long.loc[
                              shap_values_df_sample_manual_long['feature'] == 'PC1+PC2'].sort_values('all_PC2')['ecoprovince'],
                          plt.get_cmap('RdPu')(np.linspace(0.2, 0.9, n_sel)).tolist()))
PC3_color_dict = dict(zip(shap_values_df_sample_manual_long.loc[
                              shap_values_df_sample_manual_long['feature'] == 'PC1+PC2+PC3'].sort_values('all_PC3')['ecoprovince'],
                          plt.get_cmap('RdPu')(np.linspace(0.2, 0.9, n_sel)).tolist()))
FRic_gamma_color_dict = dict(zip(shap_values_df_sample_manual_long.loc[
                                     shap_values_df_sample_manual_long['feature'] == 'PC1+PC2+PC3+FRic-gamma'].sort_values(
    'FRic_gamma')['ecoprovince'],
                                 plt.get_cmap('YlGnBu')(np.linspace(0.2, 0.9, n_sel)).tolist()))
Fbeta_alpha_to_gamma_color_dict = dict(zip(shap_values_df_sample_manual_long.loc[
    shap_values_df_sample_manual_long['feature'] == 'PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma'].sort_values(
    'Fbeta_alpha_to_gamma')['ecoprovince'],
    plt.get_cmap('YlGnBu')(np.linspace(0.2, 0.9, n_sel)).tolist()))
Fbeta_gamma_to_tau_color_dict = dict(zip(shap_values_df_sample_manual_long.loc[
    shap_values_df_sample_manual_long['feature'] == 'PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma+Fbeta-gamma-to-tau'].sort_values(
    'Fbeta_gamma_to_tau')['ecoprovince'],
    plt.get_cmap('YlGnBu')(np.linspace(0.2, 0.9, n_sel)).tolist()))
FDiv_gamma_color_dict = dict(zip(shap_values_df_sample_manual_long.loc[
    shap_values_df_sample_manual_long['feature'] == 'PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma+Fbeta-gamma-to-tau+FDiv-gamma'].sort_values(
    'FDiv_gamma')['ecoprovince'],
    plt.get_cmap('YlGnBu')(np.linspace(0.2, 0.9, n_sel)).tolist()))

In [ ]:
# attach the point colour to every (ecoprovince, step) row
shap_values_df_sample_manual_long_v2 = pd.concat(
    [shap_values_df_sample_manual_long.loc[shap_values_df_sample_manual_long['feature'] == 'base'],
     pd.merge(shap_values_df_sample_manual_long.loc[shap_values_df_sample_manual_long['feature'] == 'PC1'],
              pd.DataFrame(PC1_color_dict.items(), columns=['ecoprovince', 'color']), on='ecoprovince'),
     pd.merge(
         shap_values_df_sample_manual_long.loc[shap_values_df_sample_manual_long['feature'] == 'PC1+PC2'],
         pd.DataFrame(PC2_color_dict.items(), columns=['ecoprovince', 'color']), on='ecoprovince'),
     pd.merge(
         shap_values_df_sample_manual_long.loc[shap_values_df_sample_manual_long['feature'] == 'PC1+PC2+PC3'],
         pd.DataFrame(PC3_color_dict.items(), columns=['ecoprovince', 'color']), on='ecoprovince'),
     pd.merge(
         shap_values_df_sample_manual_long.loc[shap_values_df_sample_manual_long['feature'] == 'PC1+PC2+PC3+FRic-gamma'],
         pd.DataFrame(FRic_gamma_color_dict.items(), columns=['ecoprovince', 'color']), on='ecoprovince'),
     pd.merge(
         shap_values_df_sample_manual_long.loc[shap_values_df_sample_manual_long['feature'] == 'PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma'],
         pd.DataFrame(Fbeta_alpha_to_gamma_color_dict.items(), columns=['ecoprovince', 'color']), on='ecoprovince'),
     pd.merge(
         shap_values_df_sample_manual_long.loc[shap_values_df_sample_manual_long['feature'] == 'PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma+Fbeta-gamma-to-tau'],
         pd.DataFrame(Fbeta_gamma_to_tau_color_dict.items(), columns=['ecoprovince', 'color']), on='ecoprovince'),
     pd.merge(
         shap_values_df_sample_manual_long.loc[shap_values_df_sample_manual_long['feature'] == 'PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma+Fbeta-gamma-to-tau+FDiv-gamma'],
         pd.DataFrame(FDiv_gamma_color_dict.items(), columns=['ecoprovince', 'color']), on='ecoprovince')], )

# Fig. 5d

In [ ]:
plt.style.use('default')
sns.set_style('ticks')
sns.set_context('paper')
plt.rcParams['font.family'] = ['Helvetica', 'Arial', 'DejaVu Sans']

line_cmap = 'PuOr_r'

fig, ax = plt.subplots(figsize=(4.2, 5.2 * 1.2), layout="constrained")
# one line per ecoprovince, coloured by the baseline-model residual (diverging palette with a gap around zero)
sns.lineplot(data=shap_values_df_sample_manual_long_v2, x='feature', y='SHAP', hue='Residual_null_model',
             palette=plt.get_cmap(line_cmap)(
                 np.concatenate([
                     np.linspace(0.05, 0.40, shap_values_df_sample_manual.shape[0] // 2),
                     np.linspace(0.60, 0.95, shap_values_df_sample_manual.shape[0] - shap_values_df_sample_manual.shape[0] // 2),
                 ])).tolist(),
             linewidth=1.75,
             dashes=True,
             ax=ax, legend=False)
marker_dict = {'Deciduous Forest': 'H', 'Evergreen Forest': 'o', 'Woody Wetlands': 's', 'Grassland/Herbaceous': 'd',
               'Cultivated Crops': '*', 'Shrub/Scrub': '^', }

# points: colour = rank of the added predictor, marker = NLCD class of the cell
for ecoprovince in shap_values_df_sample_manual['ecoprovince'].unique():
    for feature in ['PC1', 'PC1+PC2', 'PC1+PC2+PC3', 'PC1+PC2+PC3+FRic-gamma',
                    'PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma',
                    'PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma+Fbeta-gamma-to-tau',
                    'PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma+Fbeta-gamma-to-tau+FDiv-gamma']:
        color = shap_values_df_sample_manual_long_v2.loc[
            (shap_values_df_sample_manual_long_v2['feature'] == feature) & (
                    shap_values_df_sample_manual_long_v2['ecoprovince'] == ecoprovince), 'color'].values[0]

        ax.scatter(x=feature, y='SHAP', data=shap_values_df_sample_manual_long_v2.loc[
            (shap_values_df_sample_manual_long_v2['feature'] == feature) & (
                    shap_values_df_sample_manual_long_v2['ecoprovince'] == ecoprovince)], color=color, s=50, alpha=1,
                   marker=shap_values_df_sample_manual.loc[
                       shap_values_df_sample_manual['ecoprovince'] == ecoprovince, 'nlcd_majority'].map(
                       marker_dict).values[0], label=None,
                   linewidths=0.7, edgecolors='black',
                   zorder=10)

# label each line with its ecoprovince code
for i, row in shap_values_df_sample_manual.iterrows():
    ax.annotate(row['ecoprovince'], (7, row['SHAP_PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma+Fbeta-gamma-to-tau+FDiv-gamma']), fontsize=8,
                xytext=(7.11, row['SHAP_PC1+PC2+PC3+FRic-gamma+Fbeta-alpha-to-gamma+Fbeta-gamma-to-tau+FDiv-gamma']))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.tick_params(axis='x', direction='out', length=0)
ax.tick_params(axis='y', direction='out', length=0)
ax.grid(axis='both', linestyle='--', alpha=0.9, linewidth=1.5)

ax.set_xticks([0, 1, 2, 3, 4, 5, 6, 7])
ax.set_xticklabels([
    'Baseline model',
    '+PC1',
    '+PC2',
    r'+PC3',
    r'+FRichness-$\gamma$',
    r'+FDissimilarity-$\beta_{\alpha\to\gamma}$',
    r'+FDissimilarity-$\beta_{\gamma\to\tau}$',
    r'+FDivergence-$\gamma$',
])
ax.tick_params(axis='x', rotation=25)

# nudge the three short PC labels down so their right ends line up with the longer ones
for i in [1, 2, 3]:
    ax.get_xticklabels()[i].set_y(-0.03)

ax.set_ylabel('Model output GPP', labelpad=2)
ax.set_xlabel('SHAP value of predictor')

ax.set_xlim(-0.02, ax.get_xlim()[1])

# legend swatches (text labels were added in the figure layout): lines = baseline-model residual,
# RdPu points = functional composition, YlGnBu points = functional diversity
for i, color in enumerate(plt.get_cmap(line_cmap)(np.concatenate([np.linspace(0.05, 0.40, 3), np.linspace(0.60, 0.95, 3)])).tolist()):
    ax.plot([], [], color=color, linewidth=1.5, label=' ')
for i, color in enumerate(plt.get_cmap('RdPu')(np.linspace(0.2, 0.9, 6)).tolist()):
    ax.scatter([], [], color=color, linewidth=0.7, s=30, label=' ')
for i, color in enumerate(plt.get_cmap('YlGnBu')(np.linspace(0.2, 0.9, 6)).tolist()):
    ax.scatter([], [], color=color, linewidth=0.7, s=30, label=' ')

ax.legend(loc='upper left', bbox_to_anchor=(0.01, 0.96), frameon=False, ncols=3, handletextpad=0.,
          columnspacing=1, labelspacing=0.2, fontsize=6)

fig.get_layout_engine().set(w_pad=2 / 72, h_pad=2 / 72, hspace=2 / 72, wspace=2 / 72)

fig.savefig(f'{RESULTS_DIR}/fig5d_decision_plots.png', dpi=1000, bbox_inches='tight')
plt.show()